<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/Hybrid_emulator4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')

In [ ]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# train_gnn_1layer_multitime_stateplusphysics.py
# ------------------------------------------------------------
# Goal:
#   Train a reliable ML emulator for use inside ML-4DVAR,
#   emphasizing trajectory closeness to the FD model.
#
# Uses:
#   - auto-detected snapshot root
#   - CollocBank multi-time random collocation
#   - macro rollout supervision
#   - stronger collocation state supervision
#   - collocation tendency supervision
#   - nonlinear flux continuity
#   - nonlinear advective momentum
#   - normalized local total-energy tendency mismatch
#   - weak PV / vorticity regularization
#
# State:
#   x = [eta, u, v]
#
# Physics:
#   h = H + eta
#
#   Continuity:
#       h_t + d_x(h u) + d_y(h v) = 0
#
#   Momentum:
#       u_t + u u_x + v u_y - f v + g h_x = 0
#       v_t + u v_x + v v_y + f u + g h_y = 0
#
#   Total energy density:
#       E = 0.5 h (u^2 + v^2) + 0.5 g eta^2
#
#   Total energy tendency:
#       E_t = 0.5 (u^2+v^2) h_t + h (u u_t + v v_t) + g eta h_t
#
# Loss stack:
#   L =
#     lambda_data       * macro rollout loss
#   + lambda_coll_state * collocation state loss
#   + lambda_coll_tend  * collocation tendency loss
#   + lambda_cont       * continuity residual
#   + lambda_mom        * momentum residual
#   + lambda_energy     * normalized energy tendency mismatch
#   + lambda_pv         * PV mismatch
#   + lambda_vort       * vorticity mismatch
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_multitime_stateplusphysics"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Domain / physics
# ------------------------------------------------------------
NX = 256
NY = 128
Lx = 2.0e7
Ly = 8.0e6
DX = Lx / NX
DY = Ly / NY

H = 1000.0
G = 9.81
DT_MACRO = 200.0 * 30.0   # 6000 s
EPS = 1e-6
H_MIN = 10.0

# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
BATCH_SIZE = 1
EPOCHS = 12
LR = 5e-4
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 4
ROLL_GAMMA = 0.95

MAX_WINDOWS_PER_IC = 4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# ------------------------------------------------------------
# Multi-time collocation config
# ------------------------------------------------------------
N_TIME_SLICES = 4
PTS_PER_TIME = 4

# ------------------------------------------------------------
# Loss weights
# These are chosen to make state collocation matter more.
# ------------------------------------------------------------
LAMBDA_DATA        = 1.0
LAMBDA_COLL_STATE  = 0.05   # stronger than before
LAMBDA_COLL_TEND   = 0.5
LAMBDA_FLUX_CONT   = 1.0
LAMBDA_MOM_NL      = 2.0
LAMBDA_ENERGY      = 2.0
LAMBDA_PV          = 0.05

USE_VORTICITY_TERM = True
LAMBDA_VORT        = 0.02

# energy stabilization
ENERGY_DENOM_FLOOR = 1e-4
ENERGY_CLIP_MAX    = 20.0

SAVE_EVERY_EPOCH = 1
PRINT_BATCH_EVERY = 10
RESUME_FROM = None

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_flux_cont", "train_mom_nl",
            "train_energy", "train_pv", "train_vort"
        ])
        for row in loss_history:
            writer.writerow(row)

def safe_mean(x):
    if x.numel() == 0:
        return torch.tensor(0.0, device=device)
    return x.mean()

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class MultiStepContinuousPINNModel(nn.Module):
    def __init__(self, feat_ch=64, hidden=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.encoder(x)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Multi-time collocation sampler
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Physics helpers
# ------------------------------------------------------------
def total_energy_tendency(eta, u, v, h_t, u_t, v_t):
    h = H + eta
    return 0.5 * (u*u + v*v) * h_t + h * (u * u_t + v * v_t) + G * eta * h_t

# ------------------------------------------------------------
# Continuous interval losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / Lx) - 1.0
    y_norm = (2.0 * y_m / Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / Lx)
    eta_y = eta_yn * (2.0 / Ly)
    h_x   = eta_x
    h_y   = eta_y

    u_x = u_xn * (2.0 / Lx)
    u_y = u_yn * (2.0 / Ly)
    v_x = v_xn * (2.0 / Lx)
    v_y = v_yn * (2.0 / Ly)

    eta_t = eta_tau / dt_macro
    h_t   = eta_t
    u_t   = u_tau / dt_macro
    v_t   = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat

    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)

    hu_x = hu_xn * (2.0 / Lx)
    hv_y = hv_yn * (2.0 / Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    if "zeta_fd" in colloc:
        zeta_fd = torch.as_tensor(colloc["zeta_fd"], dtype=torch.float32, device=device)
    else:
        zeta_fd = None

    # 1) collocation state match
    # Main anchor to keep model close to FD trajectory between macro times
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state = loss_coll_state + ((u_hat - u_true) ** 2).mean()
    loss_coll_state = loss_coll_state + ((v_hat - v_true) ** 2).mean()

    # 2) collocation tendency match
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend = loss_coll_tend + ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend = loss_coll_tend + ((v_t - dvc_dt_fd) ** 2).mean()

    # 3) nonlinear continuity
    resid_cont = h_t + hu_x + hv_y
    loss_flux_cont = (resid_cont ** 2).mean()

    # 4) nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y

    resid_u = u_t + adv_u - f_fd * v_hat + G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + G * h_y
    loss_mom_nl = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    # 5) normalized local energy tendency mismatch
    Et_pred = total_energy_tendency(eta_hat, u_hat, v_hat, h_t, u_t, v_t)
    Et_fd   = total_energy_tendency(eta_true, u_true, v_true, deta_dt_fd, duc_dt_fd, dvc_dt_fd)

    energy_num = torch.abs(Et_pred - Et_fd)
    energy_den = torch.abs(Et_fd) + ENERGY_DENOM_FLOOR
    energy_term = energy_num / energy_den
    energy_term = torch.clamp(energy_term, max=ENERGY_CLIP_MAX)
    loss_energy = safe_mean(energy_term)

    # 6) weak PV loss
    zeta_pred = v_x - u_y
    q_pred = (zeta_pred + f_fd) / torch.clamp(h_hat, min=H_MIN)

    if zeta_fd is not None:
        q_fd = (zeta_fd + f_fd) / torch.clamp(H + eta_true, min=H_MIN)
        loss_pv = ((q_pred - q_fd) ** 2).mean()
    else:
        loss_pv = torch.tensor(0.0, device=device)

    # 7) weak vorticity loss
    if USE_VORTICITY_TERM and (zeta_fd is not None):
        loss_vort = ((zeta_pred - zeta_fd) ** 2).mean()
    else:
        loss_vort = torch.tensor(0.0, device=device)

    return (
        loss_coll_state,
        loss_coll_tend,
        loss_flux_cont,
        loss_mom_nl,
        loss_energy,
        loss_pv,
        loss_vort,
    )

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=ROLL_STEPS,
        max_windows_per_ic=MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousPINNModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")

    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_TIME_SLICES         = {N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {PTS_PER_TIME}")
    print(f"[Train] LAMBDA_DATA           = {LAMBDA_DATA}")
    print(f"[Train] LAMBDA_COLL_STATE     = {LAMBDA_COLL_STATE}")
    print(f"[Train] LAMBDA_COLL_TEND      = {LAMBDA_COLL_TEND}")
    print(f"[Train] LAMBDA_FLUX_CONT      = {LAMBDA_FLUX_CONT}")
    print(f"[Train] LAMBDA_MOM_NL         = {LAMBDA_MOM_NL}")
    print(f"[Train] LAMBDA_ENERGY         = {LAMBDA_ENERGY}")
    print(f"[Train] LAMBDA_PV             = {LAMBDA_PV}")
    print(f"[Train] USE_VORTICITY_TERM    = {USE_VORTICITY_TERM}")
    print(f"[Train] LAMBDA_VORT           = {LAMBDA_VORT}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_flux_cont = 0.0
        run_mom_nl = 0.0
        run_energy = 0.0
        run_pv = 0.0
        run_vort = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]

            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            loss_ctend = 0.0
            loss_flux_cont = 0.0
            loss_mom_nl = 0.0
            loss_energy = 0.0
            loss_pv = 0.0
            loss_vort = 0.0
            n_phys = 0

            x = seq[:, 0]

            for k in range(ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=N_TIME_SLICES,
                        pts_per_time=PTS_PER_TIME,
                    )

                    (l_cstate,
                     l_ctend,
                     l_flux_cont,
                     l_mom_nl,
                     l_energy,
                     l_pv,
                     l_vort) = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_flux_cont += l_flux_cont
                    loss_mom_nl += l_mom_nl
                    loss_energy += l_energy
                    loss_pv += l_pv
                    loss_vort += l_vort
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_flux_cont = loss_flux_cont / n_phys
                loss_mom_nl = loss_mom_nl / n_phys
                loss_energy = loss_energy / n_phys
                loss_pv = loss_pv / n_phys
                loss_vort = loss_vort / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_flux_cont = zero
                loss_mom_nl = zero
                loss_energy = zero
                loss_pv = zero
                loss_vort = zero

            loss = (
                LAMBDA_DATA * loss_data
                + LAMBDA_COLL_STATE * loss_cstate
                + LAMBDA_COLL_TEND * loss_ctend
                + LAMBDA_FLUX_CONT * loss_flux_cont
                + LAMBDA_MOM_NL * loss_mom_nl
                + LAMBDA_ENERGY * loss_energy
                + LAMBDA_PV * loss_pv
                + LAMBDA_VORT * loss_vort
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            run_total += loss.item()
            run_data += loss_data.item()
            run_cstate += loss_cstate.item()
            run_ctend += loss_ctend.item()
            run_flux_cont += loss_flux_cont.item()
            run_mom_nl += loss_mom_nl.item()
            run_energy += loss_energy.item()
            run_pv += loss_pv.item()
            run_vort += loss_vort.item()

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={loss.item():.6e} "
                    f"data={loss_data.item():.6e} "
                    f"cstate={loss_cstate.item():.6e} "
                    f"ctend={loss_ctend.item():.6e} "
                    f"cont={loss_flux_cont.item():.6e} "
                    f"mom={loss_mom_nl.item():.6e} "
                    f"energy={loss_energy.item():.6e} "
                    f"pv={loss_pv.item():.6e} "
                    f"vort={loss_vort.item():.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_flux_cont = run_flux_cont / n_batches
        ep_mom_nl = run_mom_nl / n_batches
        ep_energy = run_energy / n_batches
        ep_pv = run_pv / n_batches
        ep_vort = run_vort / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend,
            ep_flux_cont, ep_mom_nl, ep_energy, ep_pv, ep_vort
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_flux_cont:.6e} | mom={ep_mom_nl:.6e} | "
            f"energy={ep_energy:.6e} | pv={ep_pv:.6e} | vort={ep_vort:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cpu
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | samples=15616 | macr

# State only - no physics

In [ ]:
# ============================================================
# train_gnn_1layer_multitime_stateonly.py
# ------------------------------------------------------------
# Purpose:
#   Train a 1-layer emulator using ONLY state matching:
#     - macro-step rollout data loss
#     - interior space-time collocation state loss
#
# No physics losses are used here.
#
# This is intended as a clean test of how much benefit comes
# purely from matching more FD data over space and time.
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_multitime_stateonly"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Domain / timing
# ------------------------------------------------------------
NX = 256
NY = 128
Lx = 2.0e7
Ly = 8.0e6
DT_MACRO = 200.0 * 30.0  # 6000 s

# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
BATCH_SIZE = 1
EPOCHS = 12
LR = 5e-4
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 4
ROLL_GAMMA = 0.95

MAX_WINDOWS_PER_IC = 4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# ------------------------------------------------------------
# Multi-time collocation config
# ------------------------------------------------------------
N_TIME_SLICES = 4
PTS_PER_TIME = 4   # total collocation points per interval = 16

# ------------------------------------------------------------
# Loss weights
# ------------------------------------------------------------
LAMBDA_DATA       = 1.0
LAMBDA_COLL_STATE = 0.05

SAVE_EVERY_EPOCH = 1
PRINT_BATCH_EVERY = 10
RESUME_FROM = None

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_total", "train_data", "train_coll_state"
        ])
        for row in loss_history:
            writer.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class MultiStepContinuousStateModel(nn.Module):
    def __init__(self, feat_ch=64, hidden=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.encoder(x)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Multi-time collocation sampler
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Collocation state loss only
# ------------------------------------------------------------
def collocation_state_loss(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / Lx) - 1.0
    y_norm = (2.0 * y_m / DT_MACRO * 0.0 + 2.0 * y_m / Ly) - 1.0  # keep same scaling form
    y_norm = (2.0 * y_m / Ly) - 1.0

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)

    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    loss = ((eta_hat - eta_true) ** 2).mean()
    loss = loss + ((u_hat - u_true) ** 2).mean()
    loss = loss + ((v_hat - v_true) ** 2).mean()

    return loss

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=ROLL_STEPS,
        max_windows_per_ic=MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousStateModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")

    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_TIME_SLICES         = {N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {PTS_PER_TIME}")
    print(f"[Train] LAMBDA_DATA           = {LAMBDA_DATA}")
    print(f"[Train] LAMBDA_COLL_STATE     = {LAMBDA_COLL_STATE}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]

            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            n_coll = 0

            x = seq[:, 0]

            for k in range(ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=N_TIME_SLICES,
                        pts_per_time=PTS_PER_TIME,
                    )

                    l_cstate = collocation_state_loss(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    n_coll += 1

                x = x_next

            if n_coll > 0:
                loss_cstate = loss_cstate / n_coll
            else:
                loss_cstate = torch.tensor(0.0, device=device)

            loss = LAMBDA_DATA * loss_data + LAMBDA_COLL_STATE * loss_cstate

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  loss_data   =", float(loss_data))
                print("  loss_cstate =", float(loss_cstate))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.item())
            run_data += float(loss_data.item())
            run_cstate += float(loss_cstate.item())

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={loss.item():.6e} "
                    f"data={loss_data.item():.6e} "
                    f"cstate={loss_cstate.item():.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches

        loss_history.append([epoch, ep_total, ep_data, ep_cstate])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cuda
GPU: Tesla T4
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | sampl

# Evaluation script

In [ ]:
# ============================================================
# eval_gnn_1layer_multitime_stateonly.py
# ------------------------------------------------------------
# Evaluation script for:
#   train_gnn_1layer_multitime_stateonly.py
#
# Produces:
#   - RMSE vs rollout time for eta, u, v
#   - domain-mean KE vs rollout time for FD vs ML
#   - optional field plots
#   - CSV metrics
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_multitime_stateonly"
CKPT_NAME = "final_model.pt"

EVAL_DIR = os.path.join(CKPT_DIR, "eval_rollout")
os.makedirs(EVAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
DT_MACRO = 200.0 * 30.0

# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------
IC_KEY_TO_EVAL = None   # e.g. "rh4"
START_INDEX = 0
ROLLOUT_STEPS = 6
FIELD_PLOT_STEPS = [0, 2, 5]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def domain_mean_ke(eta, u, v, H=1000.0):
    h = H + eta
    ke = 0.5 * h * (u*u + v*v)
    return float(np.mean(ke))

def list_ic_dirs(data_dir):
    return sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class MultiStepContinuousStateModel(nn.Module):
    def __init__(self, feat_ch=64, hidden=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.encoder(x)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
ckpt_path = os.path.join(CKPT_DIR, CKPT_NAME)
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model = MultiStepContinuousStateModel().to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"[Eval] loaded checkpoint: {ckpt_path}")

# ------------------------------------------------------------
# Locate FD data
# ------------------------------------------------------------
data_dir = ckpt.get("data_dir", None)
if (data_dir is None) or (not os.path.exists(data_dir)):
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

print("[Eval] using FD data dir:", data_dir)

ic_dirs = list_ic_dirs(data_dir)
if len(ic_dirs) == 0:
    raise RuntimeError("No IC directories found.")

if IC_KEY_TO_EVAL is None:
    ic_dir = None
    for d in ic_dirs:
        files = sorted(glob.glob(os.path.join(d, "klein_step_*.npz")))
        n_avail = len(files) - START_INDEX - 1
        if n_avail > 0:
            ic_dir = d
            break
    if ic_dir is None:
        raise RuntimeError("Could not find any IC with usable rollout steps.")
else:
    ic_dir = os.path.join(data_dir, IC_KEY_TO_EVAL)
    if not os.path.isdir(ic_dir):
        raise FileNotFoundError(f"IC directory not found: {ic_dir}")

ic_key = os.path.basename(ic_dir)
files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

n_avail = len(files) - START_INDEX - 1
if n_avail <= 0:
    raise RuntimeError(f"No rollout steps available in {ic_key}.")

if ROLLOUT_STEPS > n_avail:
    print(f"[Eval] requested ROLLOUT_STEPS={ROLLOUT_STEPS}, but only {n_avail} available. Using {n_avail}.")
    ROLLOUT_STEPS = n_avail

print("[Eval] IC:", ic_key)
print("[Eval] number of files:", len(files))
print("[Eval] rollout steps:", ROLLOUT_STEPS)

# ------------------------------------------------------------
# Load truth
# ------------------------------------------------------------
truth_seq = []
truth_times = []
truth_steps = []

for f in files[START_INDEX:START_INDEX + ROLLOUT_STEPS + 1]:
    z = np.load(f)
    truth_seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
    truth_times.append(float(z["t"]))
    truth_steps.append(int(extract_step_from_path(f)))

truth_seq = np.stack(truth_seq, axis=0)
truth_times = np.array(truth_times, dtype=np.float32)
truth_steps = np.array(truth_steps, dtype=np.int32)

# ------------------------------------------------------------
# Rollout
# ------------------------------------------------------------
x = torch.tensor(truth_seq[0], dtype=torch.float32, device=device).unsqueeze(0)
pred_seq = [truth_seq[0].copy()]

with torch.no_grad():
    for k in range(ROLLOUT_STEPS):
        feat = model.encode(x)
        xdot = model.grid_tendency(feat)
        x = x + DT_MACRO * xdot
        pred_seq.append(x[0].detach().cpu().numpy())

pred_seq = np.stack(pred_seq, axis=0)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------
rmse_eta = []
rmse_u = []
rmse_v = []

ke_fd = []
ke_ml = []

for k in range(1, ROLLOUT_STEPS + 1):
    eta_fd = truth_seq[k, 0]
    u_fd   = truth_seq[k, 1]
    v_fd   = truth_seq[k, 2]

    eta_ml = pred_seq[k, 0]
    u_ml   = pred_seq[k, 1]
    v_ml   = pred_seq[k, 2]

    rmse_eta.append(rmse(eta_ml, eta_fd))
    rmse_u.append(rmse(u_ml, u_fd))
    rmse_v.append(rmse(v_ml, v_fd))

    ke_fd.append(domain_mean_ke(eta_fd, u_fd, v_fd, H=H))
    ke_ml.append(domain_mean_ke(eta_ml, u_ml, v_ml, H=H))

rmse_eta = np.array(rmse_eta)
rmse_u   = np.array(rmse_u)
rmse_v   = np.array(rmse_v)
ke_fd    = np.array(ke_fd)
ke_ml    = np.array(ke_ml)

roll_steps = np.arange(1, ROLLOUT_STEPS + 1)
roll_hours = roll_steps * DT_MACRO / 3600.0

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------
csv_path = os.path.join(EVAL_DIR, f"{ic_key}_rollout_metrics.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["roll_step", "hours", "rmse_eta", "rmse_u", "rmse_v", "ke_fd", "ke_ml"])
    for i in range(ROLLOUT_STEPS):
        w.writerow([
            int(roll_steps[i]),
            float(roll_hours[i]),
            float(rmse_eta[i]),
            float(rmse_u[i]),
            float(rmse_v[i]),
            float(ke_fd[i]),
            float(ke_ml[i]),
        ])

# ------------------------------------------------------------
# Plot RMSE
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(roll_hours, rmse_eta, label="eta RMSE")
plt.plot(roll_hours, rmse_u, label="u RMSE")
plt.plot(roll_hours, rmse_v, label="v RMSE")
plt.xlabel("Rollout time (hours)")
plt.ylabel("RMSE")
plt.title(f"RMSE vs rollout time | {ic_key}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_rollout_rmse.png"), dpi=160)
plt.close()

# ------------------------------------------------------------
# Plot KE
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(roll_hours, ke_fd, label="FD KE")
plt.plot(roll_hours, ke_ml, label="ML KE")
plt.xlabel("Rollout time (hours)")
plt.ylabel("Domain-mean kinetic energy")
plt.title(f"Kinetic energy vs rollout time | {ic_key}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_rollout_ke.png"), dpi=160)
plt.close()

# ------------------------------------------------------------
# Field plots
# ------------------------------------------------------------
for kk in FIELD_PLOT_STEPS:
    if kk < 0 or kk >= ROLLOUT_STEPS:
        continue

    t_idx = kk + 1
    eta_fd = truth_seq[t_idx, 0]
    u_fd   = truth_seq[t_idx, 1]
    v_fd   = truth_seq[t_idx, 2]

    eta_ml = pred_seq[t_idx, 0]
    u_ml   = pred_seq[t_idx, 1]
    v_ml   = pred_seq[t_idx, 2]

    fig, axes = plt.subplots(3, 3, figsize=(14, 11))

    im = axes[0, 0].imshow(eta_fd, origin="lower")
    axes[0, 0].set_title(f"FD eta | step {t_idx}")
    plt.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

    im = axes[0, 1].imshow(eta_ml, origin="lower")
    axes[0, 1].set_title("ML eta")
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

    im = axes[0, 2].imshow(eta_ml - eta_fd, origin="lower")
    axes[0, 2].set_title("ML - FD eta")
    plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

    im = axes[1, 0].imshow(u_fd, origin="lower")
    axes[1, 0].set_title("FD u")
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

    im = axes[1, 1].imshow(u_ml, origin="lower")
    axes[1, 1].set_title("ML u")
    plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

    im = axes[1, 2].imshow(u_ml - u_fd, origin="lower")
    axes[1, 2].set_title("ML - FD u")
    plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

    im = axes[2, 0].imshow(v_fd, origin="lower")
    axes[2, 0].set_title("FD v")
    plt.colorbar(im, ax=axes[2, 0], fraction=0.046, pad=0.04)

    im = axes[2, 1].imshow(v_ml, origin="lower")
    axes[2, 1].set_title("ML v")
    plt.colorbar(im, ax=axes[2, 1], fraction=0.046, pad=0.04)

    im = axes[2, 2].imshow(v_ml - v_fd, origin="lower")
    axes[2, 2].set_title("ML - FD v")
    plt.colorbar(im, ax=axes[2, 2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_fields_step_{t_idx:03d}.png"), dpi=150)
    plt.close()

print("\n=== Final rollout summary ===")
print("IC key:", ic_key)
print("Final eta RMSE:", rmse_eta[-1])
print("Final u   RMSE:", rmse_u[-1])
print("Final v   RMSE:", rmse_v[-1])
print("Final FD KE:", ke_fd[-1])
print("Final ML KE:", ke_ml[-1])
print("KE ratio ML/FD:", ke_ml[-1] / max(ke_fd[-1], 1e-12))
print("All outputs saved in:", EVAL_DIR)

Using device: cuda
[Eval] loaded checkpoint: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_multitime_stateonly/final_model.pt
[Eval] using FD data dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Eval] IC: gauss_00
[Eval] number of files: 7
[Eval] rollout steps: 6

=== Final rollout summary ===
IC key: gauss_00
Final eta RMSE: 3.74172043800354
Final u   RMSE: 3.360262870788574
Final v   RMSE: 3.1087138652801514
Final FD KE: 90.02924346923828
Final ML KE: 10584.1796875
KE ratio ML/FD: 117.56379682471135
All outputs saved in: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_multitime_stateonly/eval_rollout


# More general eveluation scrupt

In [ ]:
# ============================================================
# eval_gnn_1layer_multitime_stateonly_select_cases.py
# ------------------------------------------------------------
# Flexible evaluation for:
#   train_gnn_1layer_multitime_stateonly.py
#
# You can choose:
#   - one IC
#   - several ICs
#   - all ICs
#
# Outputs per case:
#   - RMSE plot
#   - KE plot
#   - optional field plots
#   - metrics CSV
#
# Also writes a summary CSV over all selected cases.
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_multitime_stateonly"
CKPT_NAME = "final_model.pt"

EVAL_DIR = os.path.join(CKPT_DIR, "eval_selected_cases")
os.makedirs(EVAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
DT_MACRO = 200.0 * 30.0  # 6000 s

# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
# Choose one of these modes:
#
#   MODE = "one"
#   MODE = "list"
#   MODE = "all"
#
MODE = "list"

# Used if MODE == "one"
ONE_CASE = "gauss_00"

# Used if MODE == "list"
CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
ROLLOUT_STEPS = 6

# Optional field plots for each selected case
MAKE_FIELD_PLOTS = True
FIELD_PLOT_STEPS = [0, 2, 5]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def domain_mean_ke(eta, u, v, H=1000.0):
    h = H + eta
    ke = 0.5 * h * (u*u + v*v)
    return float(np.mean(ke))

def list_ic_dirs(data_dir):
    return sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])

def list_ic_keys(data_dir):
    return [os.path.basename(d) for d in list_ic_dirs(data_dir)]

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class MultiStepContinuousStateModel(nn.Module):
    def __init__(self, feat_ch=64, hidden=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.encoder(x)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
ckpt_path = os.path.join(CKPT_DIR, CKPT_NAME)
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model = MultiStepContinuousStateModel().to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"[Eval] loaded checkpoint: {ckpt_path}")

# ------------------------------------------------------------
# Locate FD data
# ------------------------------------------------------------
data_dir = ckpt.get("data_dir", None)
if (data_dir is None) or (not os.path.exists(data_dir)):
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

print("[Eval] using FD data dir:", data_dir)

available_keys = list_ic_keys(data_dir)
print(f"[Eval] found {len(available_keys)} ICs")
print("[Eval] available ICs:", available_keys)

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------
if MODE == "one":
    selected_cases = [ONE_CASE]
elif MODE == "list":
    selected_cases = CASE_LIST[:]
elif MODE == "all":
    selected_cases = available_keys[:]
else:
    raise ValueError(f"Unknown MODE={MODE}")

# keep only valid cases
selected_cases = [k for k in selected_cases if k in available_keys]
if len(selected_cases) == 0:
    raise RuntimeError("No valid selected cases found.")

print("[Eval] selected cases:", selected_cases)

# ------------------------------------------------------------
# Evaluate one case
# ------------------------------------------------------------
def evaluate_case(ic_key):
    ic_dir = os.path.join(data_dir, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    n_avail = len(files) - START_INDEX - 1
    if n_avail <= 0:
        print(f"[Skip] {ic_key}: no rollout steps available.")
        return None

    rollout_steps = min(ROLLOUT_STEPS, n_avail)

    truth_seq = []
    truth_times = []
    truth_steps = []

    for f in files[START_INDEX:START_INDEX + rollout_steps + 1]:
        z = np.load(f)
        truth_seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
        truth_times.append(float(z["t"]))
        truth_steps.append(int(extract_step_from_path(f)))

    truth_seq = np.stack(truth_seq, axis=0)
    truth_times = np.array(truth_times, dtype=np.float32)
    truth_steps = np.array(truth_steps, dtype=np.int32)

    # rollout
    x = torch.tensor(truth_seq[0], dtype=torch.float32, device=device).unsqueeze(0)
    pred_seq = [truth_seq[0].copy()]

    with torch.no_grad():
        for k in range(rollout_steps):
            feat = model.encode(x)
            xdot = model.grid_tendency(feat)
            x = x + DT_MACRO * xdot
            pred_seq.append(x[0].detach().cpu().numpy())

    pred_seq = np.stack(pred_seq, axis=0)

    # diagnostics
    rmse_eta = []
    rmse_u = []
    rmse_v = []
    ke_fd = []
    ke_ml = []

    for k in range(1, rollout_steps + 1):
        eta_fd = truth_seq[k, 0]
        u_fd   = truth_seq[k, 1]
        v_fd   = truth_seq[k, 2]

        eta_ml = pred_seq[k, 0]
        u_ml   = pred_seq[k, 1]
        v_ml   = pred_seq[k, 2]

        rmse_eta.append(rmse(eta_ml, eta_fd))
        rmse_u.append(rmse(u_ml, u_fd))
        rmse_v.append(rmse(v_ml, v_fd))

        ke_fd.append(domain_mean_ke(eta_fd, u_fd, v_fd, H=H))
        ke_ml.append(domain_mean_ke(eta_ml, u_ml, v_ml, H=H))

    rmse_eta = np.array(rmse_eta)
    rmse_u   = np.array(rmse_u)
    rmse_v   = np.array(rmse_v)
    ke_fd    = np.array(ke_fd)
    ke_ml    = np.array(ke_ml)

    roll_steps = np.arange(1, rollout_steps + 1)
    roll_hours = roll_steps * DT_MACRO / 3600.0

    # save per-case CSV
    csv_path = os.path.join(EVAL_DIR, f"{ic_key}_rollout_metrics.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["roll_step", "hours", "rmse_eta", "rmse_u", "rmse_v", "ke_fd", "ke_ml"])
        for i in range(rollout_steps):
            w.writerow([
                int(roll_steps[i]),
                float(roll_hours[i]),
                float(rmse_eta[i]),
                float(rmse_u[i]),
                float(rmse_v[i]),
                float(ke_fd[i]),
                float(ke_ml[i]),
            ])

    # RMSE plot
    plt.figure(figsize=(8, 5))
    plt.plot(roll_hours, rmse_eta, label="eta RMSE")
    plt.plot(roll_hours, rmse_u, label="u RMSE")
    plt.plot(roll_hours, rmse_v, label="v RMSE")
    plt.xlabel("Rollout time (hours)")
    plt.ylabel("RMSE")
    plt.title(f"RMSE vs rollout time | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_rollout_rmse.png"), dpi=160)
    plt.close()

    # KE plot
    plt.figure(figsize=(8, 5))
    plt.plot(roll_hours, ke_fd, label="FD KE")
    plt.plot(roll_hours, ke_ml, label="ML KE")
    plt.xlabel("Rollout time (hours)")
    plt.ylabel("Domain-mean kinetic energy")
    plt.title(f"Kinetic energy vs rollout time | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_rollout_ke.png"), dpi=160)
    plt.close()

    # optional field plots
    if MAKE_FIELD_PLOTS:
        for kk in FIELD_PLOT_STEPS:
            if kk < 0 or kk >= rollout_steps:
                continue

            t_idx = kk + 1
            eta_fd = truth_seq[t_idx, 0]
            u_fd   = truth_seq[t_idx, 1]
            v_fd   = truth_seq[t_idx, 2]

            eta_ml = pred_seq[t_idx, 0]
            u_ml   = pred_seq[t_idx, 1]
            v_ml   = pred_seq[t_idx, 2]

            fig, axes = plt.subplots(3, 3, figsize=(14, 11))

            im = axes[0, 0].imshow(eta_fd, origin="lower")
            axes[0, 0].set_title(f"FD eta | step {t_idx}")
            plt.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

            im = axes[0, 1].imshow(eta_ml, origin="lower")
            axes[0, 1].set_title("ML eta")
            plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

            im = axes[0, 2].imshow(eta_ml - eta_fd, origin="lower")
            axes[0, 2].set_title("ML - FD eta")
            plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

            im = axes[1, 0].imshow(u_fd, origin="lower")
            axes[1, 0].set_title("FD u")
            plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

            im = axes[1, 1].imshow(u_ml, origin="lower")
            axes[1, 1].set_title("ML u")
            plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

            im = axes[1, 2].imshow(u_ml - u_fd, origin="lower")
            axes[1, 2].set_title("ML - FD u")
            plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

            im = axes[2, 0].imshow(v_fd, origin="lower")
            axes[2, 0].set_title("FD v")
            plt.colorbar(im, ax=axes[2, 0], fraction=0.046, pad=0.04)

            im = axes[2, 1].imshow(v_ml, origin="lower")
            axes[2, 1].set_title("ML v")
            plt.colorbar(im, ax=axes[2, 1], fraction=0.046, pad=0.04)

            im = axes[2, 2].imshow(v_ml - v_fd, origin="lower")
            axes[2, 2].set_title("ML - FD v")
            plt.colorbar(im, ax=axes[2, 2], fraction=0.046, pad=0.04)

            plt.tight_layout()
            plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_fields_step_{t_idx:03d}.png"), dpi=150)
            plt.close()

    # summary row
    summary = {
        "ic_key": ic_key,
        "rollout_steps": rollout_steps,
        "final_eta_rmse": float(rmse_eta[-1]),
        "final_u_rmse": float(rmse_u[-1]),
        "final_v_rmse": float(rmse_v[-1]),
        "final_fd_ke": float(ke_fd[-1]),
        "final_ml_ke": float(ke_ml[-1]),
        "final_ke_ratio_ml_over_fd": float(ke_ml[-1] / max(ke_fd[-1], 1e-12)),
        "max_eta_rmse": float(rmse_eta.max()),
        "max_u_rmse": float(rmse_u.max()),
        "max_v_rmse": float(rmse_v.max()),
    }
    return summary

# ------------------------------------------------------------
# Run selected cases
# ------------------------------------------------------------
all_summaries = []

for ic_key in selected_cases:
    print(f"\n[Eval] running case: {ic_key}")
    result = evaluate_case(ic_key)
    if result is not None:
        all_summaries.append(result)
        print("  final eta RMSE:", result["final_eta_rmse"])
        print("  final u   RMSE:", result["final_u_rmse"])
        print("  final v   RMSE:", result["final_v_rmse"])
        print("  final KE ratio:", result["final_ke_ratio_ml_over_fd"])

# ------------------------------------------------------------
# Save summary CSV
# ------------------------------------------------------------
if len(all_summaries) > 0:
    summary_csv = os.path.join(EVAL_DIR, "summary_selected_cases.csv")
    keys = list(all_summaries[0].keys())
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in all_summaries:
            w.writerow(row)

    print("\n[Eval] wrote summary CSV:", summary_csv)
else:
    print("\n[Eval] no cases were successfully evaluated.")

Using device: cuda
[Eval] loaded checkpoint: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_multitime_stateonly/final_model.pt
[Eval] using FD data dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Eval] found 28 ICs
[Eval] available ICs: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
[Eval] selected cases: ['gauss_00', 'test_modon_00', 'test_rh_00', 'wave_00']

[Eval] running case: gauss_00
  final eta RMSE: 3.74172043800354
  final u   RMSE: 3.360262870788574
  final v   RMSE: 3.1087138652801514
  final KE ratio: 117.56379682471135

[Eval] running case: test_modon_00
  final eta RMSE: 18.223936080932617
  final u   RMSE: 4.151055812835693
  final v   RMSE: 3.7929461002349854
  final KE ratio: 2.2633511443

# Evaluation with diffusion (in inference mode)

In [ ]:
# ============================================================
# eval_gnn_1layer_stateonly_with_uv_diffusion_FULL.py
# ------------------------------------------------------------
# Evaluate the state-only trained model with optional
# inference-stage horizontal diffusion applied to u and v only.
#
# This is useful to test whether simple diffusion during
# rollout can suppress the unphysical KE growth.
#
# Uses:
#   - same full model architecture as training
#   - selectable cases
#   - selectable diffusion coefficients
#
# Outputs:
#   - per-case RMSE comparison plots
#   - per-case KE comparison plots
#   - per-case metrics CSVs
#   - one summary CSV over all selected cases and all nu values
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_multitime_stateonly"
CKPT_NAME = "final_model.pt"

EVAL_DIR = os.path.join(CKPT_DIR, "eval_uv_diffusion")
os.makedirs(EVAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
DT_MACRO = 200.0 * 30.0   # 6000 s

Lx = 2.0e7
Ly = 8.0e6
NX = 256
NY = 128
DX = Lx / NX
DY = Ly / NY

# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
# MODE can be:
#   "one"  -> evaluate ONE_CASE only
#   "list" -> evaluate CASE_LIST
#   "all"  -> evaluate all available cases
MODE = "list"

ONE_CASE = "gauss_00"

CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
ROLLOUT_STEPS = 6

MAKE_FIELD_PLOTS = False
FIELD_PLOT_STEPS = [0, 2, 5]

# Diffusion coefficients to test
NU_LIST = [0.0, 5.0e4, 1.0e5, 2.0e5]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def domain_mean_ke(eta, u, v, H=1000.0):
    h = H + eta
    ke = 0.5 * h * (u*u + v*v)
    return float(np.mean(ke))

def list_ic_dirs(data_dir):
    return sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])

def list_ic_keys(data_dir):
    return [os.path.basename(d) for d in list_ic_dirs(data_dir)]

# ------------------------------------------------------------
# Discrete Laplacian for inference diffusion
# periodic in x, one-sided/nonperiodic in y
# ------------------------------------------------------------
def laplacian_np(field, dx, dy):
    # field shape [ny, nx]
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    d2x = (left - 2.0 * field + right) / (dx * dx)

    d2y = np.zeros_like(field)
    d2y[1:-1, :] = (field[2:, :] - 2.0 * field[1:-1, :] + field[:-2, :]) / (dy * dy)
    d2y[0, :]    = (field[1, :] - field[0, :]) / (dy * dy)
    d2y[-1, :]   = (field[-2, :] - field[-1, :]) / (dy * dy)

    return d2x + d2y

# ------------------------------------------------------------
# FULL model class from training
# ------------------------------------------------------------
class MultiStepContinuousStateModel(nn.Module):
    def __init__(self, feat_ch=64, hidden=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        # Included so checkpoint loads correctly
        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.encoder(x)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
ckpt_path = os.path.join(CKPT_DIR, CKPT_NAME)
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model = MultiStepContinuousStateModel().to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"[Eval] loaded checkpoint: {ckpt_path}")

# ------------------------------------------------------------
# Locate FD data
# ------------------------------------------------------------
data_dir = ckpt.get("data_dir", None)
if (data_dir is None) or (not os.path.exists(data_dir)):
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

print("[Eval] using FD data dir:", data_dir)

available_keys = list_ic_keys(data_dir)
print(f"[Eval] found {len(available_keys)} ICs")
print("[Eval] available ICs:", available_keys)

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------
if MODE == "one":
    selected_cases = [ONE_CASE]
elif MODE == "list":
    selected_cases = CASE_LIST[:]
elif MODE == "all":
    selected_cases = available_keys[:]
else:
    raise ValueError(f"Unknown MODE={MODE}")

selected_cases = [k for k in selected_cases if k in available_keys]
if len(selected_cases) == 0:
    raise RuntimeError("No valid selected cases found.")

print("[Eval] selected cases:", selected_cases)
print("[Eval] diffusion coefficients:", NU_LIST)

# ------------------------------------------------------------
# Evaluate one case for one diffusion coefficient
# ------------------------------------------------------------
def rollout_case_with_diffusion(ic_key, nu):
    ic_dir = os.path.join(data_dir, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    n_avail = len(files) - START_INDEX - 1
    if n_avail <= 0:
        print(f"[Skip] {ic_key}: no rollout steps available.")
        return None

    rollout_steps = min(ROLLOUT_STEPS, n_avail)

    truth_seq = []
    truth_times = []
    truth_steps = []

    for f in files[START_INDEX:START_INDEX + rollout_steps + 1]:
        z = np.load(f)
        truth_seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
        truth_times.append(float(z["t"]))
        truth_steps.append(int(extract_step_from_path(f)))

    truth_seq = np.stack(truth_seq, axis=0)
    truth_times = np.array(truth_times, dtype=np.float32)
    truth_steps = np.array(truth_steps, dtype=np.int32)

    # rollout
    x = torch.tensor(truth_seq[0], dtype=torch.float32, device=device).unsqueeze(0)
    pred_seq = [truth_seq[0].copy()]

    with torch.no_grad():
        for k in range(rollout_steps):
            feat = model.encode(x)
            xdot = model.grid_tendency(feat)

            x_np = x[0].detach().cpu().numpy()        # [3,ny,nx]
            xdot_np = xdot[0].detach().cpu().numpy()  # [3,ny,nx]

            eta = x_np[0]
            u   = x_np[1]
            v   = x_np[2]

            eta_dot = xdot_np[0]
            u_dot   = xdot_np[1]
            v_dot   = xdot_np[2]

            # inference-only diffusion on u and v
            if nu != 0.0:
                u_dot = u_dot + nu * laplacian_np(u, DX, DY)
                v_dot = v_dot + nu * laplacian_np(v, DX, DY)

            x_next_np = np.empty_like(x_np)
            x_next_np[0] = eta + DT_MACRO * eta_dot
            x_next_np[1] = u   + DT_MACRO * u_dot
            x_next_np[2] = v   + DT_MACRO * v_dot

            x = torch.tensor(x_next_np, dtype=torch.float32, device=device).unsqueeze(0)
            pred_seq.append(x_next_np.copy())

    pred_seq = np.stack(pred_seq, axis=0)

    # diagnostics
    rmse_eta = []
    rmse_u = []
    rmse_v = []
    ke_fd = []
    ke_ml = []

    for k in range(1, rollout_steps + 1):
        eta_fd = truth_seq[k, 0]
        u_fd   = truth_seq[k, 1]
        v_fd   = truth_seq[k, 2]

        eta_ml = pred_seq[k, 0]
        u_ml   = pred_seq[k, 1]
        v_ml   = pred_seq[k, 2]

        rmse_eta.append(rmse(eta_ml, eta_fd))
        rmse_u.append(rmse(u_ml, u_fd))
        rmse_v.append(rmse(v_ml, v_fd))

        ke_fd.append(domain_mean_ke(eta_fd, u_fd, v_fd, H=H))
        ke_ml.append(domain_mean_ke(eta_ml, u_ml, v_ml, H=H))

    rmse_eta = np.array(rmse_eta)
    rmse_u   = np.array(rmse_u)
    rmse_v   = np.array(rmse_v)
    ke_fd    = np.array(ke_fd)
    ke_ml    = np.array(ke_ml)

    roll_steps = np.arange(1, rollout_steps + 1)
    roll_hours = roll_steps * DT_MACRO / 3600.0

    return {
        "ic_key": ic_key,
        "nu": nu,
        "rollout_steps": rollout_steps,
        "roll_hours": roll_hours,
        "truth_seq": truth_seq,
        "pred_seq": pred_seq,
        "rmse_eta": rmse_eta,
        "rmse_u": rmse_u,
        "rmse_v": rmse_v,
        "ke_fd": ke_fd,
        "ke_ml": ke_ml,
    }

# ------------------------------------------------------------
# Run all selected cases and coefficients
# ------------------------------------------------------------
summary_rows = []

for ic_key in selected_cases:
    print(f"\n[Eval] case: {ic_key}")

    case_results = []
    for nu in NU_LIST:
        result = rollout_case_with_diffusion(ic_key, nu)
        if result is None:
            continue
        case_results.append(result)

        summary_rows.append({
            "ic_key": ic_key,
            "nu": nu,
            "final_eta_rmse": float(result["rmse_eta"][-1]),
            "final_u_rmse": float(result["rmse_u"][-1]),
            "final_v_rmse": float(result["rmse_v"][-1]),
            "final_fd_ke": float(result["ke_fd"][-1]),
            "final_ml_ke": float(result["ke_ml"][-1]),
            "final_ke_ratio": float(result["ke_ml"][-1] / max(result["ke_fd"][-1], 1e-12)),
            "max_eta_rmse": float(result["rmse_eta"].max()),
            "max_u_rmse": float(result["rmse_u"].max()),
            "max_v_rmse": float(result["rmse_v"].max()),
        })

        # save per-case per-nu CSV
        per_csv = os.path.join(EVAL_DIR, f"{ic_key}_nu_{nu:.1e}_metrics.csv")
        with open(per_csv, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["roll_step", "hours", "rmse_eta", "rmse_u", "rmse_v", "ke_fd", "ke_ml"])
            for i in range(result["rollout_steps"]):
                w.writerow([
                    int(i + 1),
                    float(result["roll_hours"][i]),
                    float(result["rmse_eta"][i]),
                    float(result["rmse_u"][i]),
                    float(result["rmse_v"][i]),
                    float(result["ke_fd"][i]),
                    float(result["ke_ml"][i]),
                ])

        print(
            f"  nu={nu:.3e} | "
            f"final eta={result['rmse_eta'][-1]:.4f} | "
            f"final u={result['rmse_u'][-1]:.4f} | "
            f"final v={result['rmse_v'][-1]:.4f} | "
            f"KE ratio={result['ke_ml'][-1] / max(result['ke_fd'][-1], 1e-12):.4e}"
        )

    if len(case_results) == 0:
        continue

    # RMSE comparison plot
    plt.figure(figsize=(9, 5))
    for result in case_results:
        nu = result["nu"]
        hrs = result["roll_hours"]
        total_rmse = np.sqrt(result["rmse_eta"]**2 + result["rmse_u"]**2 + result["rmse_v"]**2)
        plt.plot(hrs, total_rmse, label=f"nu={nu:.1e}")
    plt.xlabel("Rollout time (hours)")
    plt.ylabel("Combined RMSE")
    plt.title(f"RMSE comparison with u,v diffusion | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_rmse_compare_diffusion.png"), dpi=160)
    plt.close()

    # KE comparison plot
    plt.figure(figsize=(9, 5))
    plt.plot(case_results[0]["roll_hours"], case_results[0]["ke_fd"], label="FD KE", linewidth=2)
    for result in case_results:
        nu = result["nu"]
        hrs = result["roll_hours"]
        plt.plot(hrs, result["ke_ml"], label=f"ML KE, nu={nu:.1e}")
    plt.xlabel("Rollout time (hours)")
    plt.ylabel("Domain-mean kinetic energy")
    plt.title(f"KE comparison with u,v diffusion | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_ke_compare_diffusion.png"), dpi=160)
    plt.close()

    # Optional field plots for largest diffusion only
    if MAKE_FIELD_PLOTS:
        best_nu = NU_LIST[-1]
        best_result = None
        for result in case_results:
            if result["nu"] == best_nu:
                best_result = result
                break

        if best_result is not None:
            truth_seq = best_result["truth_seq"]
            pred_seq = best_result["pred_seq"]

            for kk in FIELD_PLOT_STEPS:
                if kk < 0 or kk >= best_result["rollout_steps"]:
                    continue

                t_idx = kk + 1
                eta_fd = truth_seq[t_idx, 0]
                u_fd   = truth_seq[t_idx, 1]
                v_fd   = truth_seq[t_idx, 2]

                eta_ml = pred_seq[t_idx, 0]
                u_ml   = pred_seq[t_idx, 1]
                v_ml   = pred_seq[t_idx, 2]

                fig, axes = plt.subplots(3, 3, figsize=(14, 11))

                im = axes[0, 0].imshow(eta_fd, origin="lower")
                axes[0, 0].set_title(f"FD eta | step {t_idx}")
                plt.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

                im = axes[0, 1].imshow(eta_ml, origin="lower")
                axes[0, 1].set_title(f"ML eta | nu={best_nu:.1e}")
                plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

                im = axes[0, 2].imshow(eta_ml - eta_fd, origin="lower")
                axes[0, 2].set_title("ML - FD eta")
                plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

                im = axes[1, 0].imshow(u_fd, origin="lower")
                axes[1, 0].set_title("FD u")
                plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

                im = axes[1, 1].imshow(u_ml, origin="lower")
                axes[1, 1].set_title("ML u")
                plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

                im = axes[1, 2].imshow(u_ml - u_fd, origin="lower")
                axes[1, 2].set_title("ML - FD u")
                plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

                im = axes[2, 0].imshow(v_fd, origin="lower")
                axes[2, 0].set_title("FD v")
                plt.colorbar(im, ax=axes[2, 0], fraction=0.046, pad=0.04)

                im = axes[2, 1].imshow(v_ml, origin="lower")
                axes[2, 1].set_title("ML v")
                plt.colorbar(im, ax=axes[2, 1], fraction=0.046, pad=0.04)

                im = axes[2, 2].imshow(v_ml - v_fd, origin="lower")
                axes[2, 2].set_title("ML - FD v")
                plt.colorbar(im, ax=axes[2, 2], fraction=0.046, pad=0.04)

                plt.tight_layout()
                plt.savefig(
                    os.path.join(EVAL_DIR, f"{ic_key}_fields_step_{t_idx:03d}_nu_{best_nu:.1e}.png"),
                    dpi=150
                )
                plt.close()

# ------------------------------------------------------------
# Save summary CSV
# ------------------------------------------------------------
if len(summary_rows) > 0:
    summary_csv = os.path.join(EVAL_DIR, "summary_uv_diffusion.csv")
    keys = list(summary_rows[0].keys())
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in summary_rows:
            w.writerow(row)
    print("\n[Eval] wrote summary CSV:", summary_csv)
else:
    print("\n[Eval] no successful evaluations.")

Using device: cuda
GPU: Tesla T4
[Eval] loaded checkpoint: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_multitime_stateonly/final_model.pt
[Eval] using FD data dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Eval] found 28 ICs
[Eval] available ICs: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
[Eval] selected cases: ['gauss_00', 'test_modon_00', 'test_rh_00', 'wave_00']
[Eval] diffusion coefficients: [0.0, 50000.0, 100000.0, 200000.0]

[Eval] case: gauss_00
  nu=0.000e+00 | final eta=3.7417 | final u=3.3603 | final v=3.1087 | KE ratio=1.1756e+02
  nu=5.000e+04 | final eta=3.7229 | final u=3.2230 | final v=2.8599 | KE ratio=1.0424e+02
  nu=1.000e+05 | final eta=3.7075 | final u=3.2195 | final v=2.7714 |

# Evaluation with geostrophic nudging

# A deeper residual CNN

In [ ]:
# ============================================================
# train_rescnn_1layer_multitime_stateplusphysics.py
# ------------------------------------------------------------
# 1-layer shallow-water emulator on Klein beta-plane
#
# Model:
#   Deeper residual CNN that predicts macro-step tendency
#
# Training losses:
#   - multistep rollout data loss
#   - collocation state loss on (eta, u, v)
#   - collocation tendency loss
#   - nonlinear continuity residual
#   - nonlinear momentum residual
#
# No KE/PV/energy losses in this version.
#
# Purpose:
#   Test whether a stronger residual CNN architecture plus
#   state collocation and basic physics improves rollout
#   stability before reintroducing more specialized constraints.
#
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_rescnn_multitime_stateplusphysics"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Domain / physics constants
# ------------------------------------------------------------
NX = 256
NY = 128
Lx = 2.0e7
Ly = 8.0e6
DX = Lx / NX
DY = Ly / NY

H = 1000.0
G = 9.81
DT_MACRO = 200.0 * 30.0   # 6000 s

# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
BATCH_SIZE = 1
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 4
ROLL_GAMMA = 0.95

MAX_WINDOWS_PER_IC = 4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# ------------------------------------------------------------
# Multi-time collocation config
# ------------------------------------------------------------
N_TIME_SLICES = 4
PTS_PER_TIME = 4

# ------------------------------------------------------------
# Loss weights
# ------------------------------------------------------------
LAMBDA_DATA       = 1.0
LAMBDA_COLL_STATE = 0.05
LAMBDA_COLL_TEND  = 0.5
LAMBDA_CONT       = 1.0
LAMBDA_MOM        = 2.0

SAVE_EVERY_EPOCH = 1
PRINT_BATCH_EVERY = 10
RESUME_FROM = None

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom"
        ])
        for row in loss_history:
            writer.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Residual CNN model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, kernel_size=3, padding=1),
        )

        # kept for collocation query
        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        x = self.stem(x)
        x = self.resnet(x)
        return x

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Multi-time collocation sampler
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / Lx) - 1.0
    y_norm = (2.0 * y_m / Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / Lx)
    eta_y = eta_yn * (2.0 / Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / Lx)
    u_y = u_yn * (2.0 / Ly)
    v_x = v_xn * (2.0 / Lx)
    v_y = v_yn * (2.0 / Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat

    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)

    hu_x = hu_xn * (2.0 / Lx)
    hv_y = hv_yn * (2.0 / Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    # 1) state collocation
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state = loss_coll_state + ((u_hat - u_true) ** 2).mean()
    loss_coll_state = loss_coll_state + ((v_hat - v_true) ** 2).mean()

    # 2) tendency collocation
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend = loss_coll_tend + ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend = loss_coll_tend + ((v_t - dvc_dt_fd) ** 2).mean()

    # 3) continuity in flux form
    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    # 4) nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y

    resid_u = u_t + adv_u - f_fd * v_hat + G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=ROLL_STEPS,
        max_windows_per_ic=MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")

    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] LAMBDA_DATA           = {LAMBDA_DATA}")
    print(f"[Train] LAMBDA_COLL_STATE     = {LAMBDA_COLL_STATE}")
    print(f"[Train] LAMBDA_COLL_TEND      = {LAMBDA_COLL_TEND}")
    print(f"[Train] LAMBDA_CONT           = {LAMBDA_CONT}")
    print(f"[Train] LAMBDA_MOM            = {LAMBDA_MOM}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            loss_ctend = 0.0
            loss_cont = 0.0
            loss_mom = 0.0
            n_phys = 0

            x = seq[:, 0]

            for k in range(ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=N_TIME_SLICES,
                        pts_per_time=PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero

            loss = (
                LAMBDA_DATA       * loss_data
                + LAMBDA_COLL_STATE * loss_cstate
                + LAMBDA_COLL_TEND  * loss_ctend
                + LAMBDA_CONT       * loss_cont
                + LAMBDA_MOM        * loss_mom
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  loss_data   =", float(loss_data))
                print("  loss_cstate =", float(loss_cstate))
                print("  loss_ctend  =", float(loss_ctend))
                print("  loss_cont   =", float(loss_cont))
                print("  loss_mom    =", float(loss_mom))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.item())
            run_data += float(loss_data.item())
            run_cstate += float(loss_cstate.item())
            run_ctend += float(loss_ctend.item())
            run_cont += float(loss_cont.item())
            run_mom += float(loss_mom.item())

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={loss.item():.6e} "
                    f"data={loss_data.item():.6e} "
                    f"cstate={loss_cstate.item():.6e} "
                    f"ctend={loss_ctend.item():.6e} "
                    f"cont={loss_cont.item():.6e} "
                    f"mom={loss_mom.item():.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend, ep_cont, ep_mom
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cpu
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | samples=15616 | macr

/tmp/ipykernel_5564/3643156514.py:613: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("  loss_data   =", float(loss_data))


RuntimeError: Non-finite loss at epoch=0, batch=36: inf

# Stable version

In [ ]:
# ============================================================
# train_rescnn_1layer_multitime_stateplusphysics_stable.py
# ------------------------------------------------------------
# 1-layer shallow-water emulator on Klein beta-plane
#
# Model:
#   Deeper residual CNN that predicts macro-step tendency
#
# Training losses:
#   - multistep rollout data loss
#   - collocation state loss on (eta, u, v)
#   - collocation tendency loss
#   - nonlinear continuity residual
#   - nonlinear momentum residual
#
# Stabilization changes:
#   - zero-init final tendency head
#   - zero-init final query MLP layer
#   - lower LR
#   - weaker initial physics weights
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_rescnn_multitime_stateplusphysics_stable"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Domain / physics constants
# ------------------------------------------------------------
NX = 256
NY = 128
Lx = 2.0e7
Ly = 8.0e6
DX = Lx / NX
DY = Ly / NY

H = 1000.0
G = 9.81
DT_MACRO = 200.0 * 30.0   # 6000 s

# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
BATCH_SIZE = 1
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 4
ROLL_GAMMA = 0.95

MAX_WINDOWS_PER_IC = 4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# ------------------------------------------------------------
# Multi-time collocation config
# ------------------------------------------------------------
N_TIME_SLICES = 4
PTS_PER_TIME = 4

# ------------------------------------------------------------
# Loss weights
# ------------------------------------------------------------
LAMBDA_DATA       = 1.0
LAMBDA_COLL_STATE = 0.05
LAMBDA_COLL_TEND  = 0.1
LAMBDA_CONT       = 0.2
LAMBDA_MOM        = 0.5

SAVE_EVERY_EPOCH = 1
PRINT_BATCH_EVERY = 10
RESUME_FROM = None

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom"
        ])
        for row in loss_history:
            writer.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Residual CNN model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, kernel_size=3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        # ---- critical stabilization ----
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        x = self.stem(x)
        x = self.resnet(x)
        return x

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Multi-time collocation sampler
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / Lx) - 1.0
    y_norm = (2.0 * y_m / Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / Lx)
    eta_y = eta_yn * (2.0 / Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / Lx)
    u_y = u_yn * (2.0 / Ly)
    v_x = v_xn * (2.0 / Lx)
    v_y = v_yn * (2.0 / Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat

    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)

    hu_x = hu_xn * (2.0 / Lx)
    hv_y = hv_yn * (2.0 / Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state = loss_coll_state + ((u_hat - u_true) ** 2).mean()
    loss_coll_state = loss_coll_state + ((v_hat - v_true) ** 2).mean()

    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend = loss_coll_tend + ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend = loss_coll_tend + ((v_t - dvc_dt_fd) ** 2).mean()

    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y

    resid_u = u_t + adv_u - f_fd * v_hat + G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=ROLL_STEPS,
        max_windows_per_ic=MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")

    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] LAMBDA_DATA           = {LAMBDA_DATA}")
    print(f"[Train] LAMBDA_COLL_STATE     = {LAMBDA_COLL_STATE}")
    print(f"[Train] LAMBDA_COLL_TEND      = {LAMBDA_COLL_TEND}")
    print(f"[Train] LAMBDA_CONT           = {LAMBDA_CONT}")
    print(f"[Train] LAMBDA_MOM            = {LAMBDA_MOM}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            loss_ctend = 0.0
            loss_cont = 0.0
            loss_mom = 0.0
            n_phys = 0

            x = seq[:, 0]

            for k in range(ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=N_TIME_SLICES,
                        pts_per_time=PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero

            loss = (
                LAMBDA_DATA         * loss_data
                + LAMBDA_COLL_STATE * loss_cstate
                + LAMBDA_COLL_TEND  * loss_ctend
                + LAMBDA_CONT       * loss_cont
                + LAMBDA_MOM        * loss_mom
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  loss_data   =", float(loss_data.detach().cpu()))
                print("  loss_cstate =", float(loss_cstate.detach().cpu()))
                print("  loss_ctend  =", float(loss_ctend.detach().cpu()))
                print("  loss_cont   =", float(loss_cont.detach().cpu()))
                print("  loss_mom    =", float(loss_mom.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.detach().cpu())
            run_data += float(loss_data.detach().cpu())
            run_cstate += float(loss_cstate.detach().cpu())
            run_ctend += float(loss_ctend.detach().cpu())
            run_cont += float(loss_cont.detach().cpu())
            run_mom += float(loss_mom.detach().cpu())

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} "
                    f"data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} "
                    f"ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} "
                    f"mom={float(loss_mom.detach().cpu()):.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend, ep_cont, ep_mom
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cuda
GPU: Tesla T4
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | sampl

# Evaluation matching the test

In [ ]:
# ============================================================
# eval_rescnn_1layer_multitime_stateplusphysics_FULL.py
# ------------------------------------------------------------
# Evaluation script for:
#   train_rescnn_1layer_multitime_stateplusphysics_stable.py
#
# Features:
#   - exact residual CNN architecture matching training
#   - selectable cases
#   - plain rollout
#   - rollout + u,v diffusion
#   - rollout + weak geostrophic nudging
#   - summary CSV + per-case plots
#
# All outputs go to:
#   /content/drive/MyDrive/AI_4DVAR2/...
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR_CANDIDATES = [
    f"{ROOT_IN}/klein_ckpt_1L",
    f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
    f"{ROOT_IN}/klein_ckpt_AL_centers",
    f"{ROOT_IN}/klein_ckpt_1L_colloc",
]

CKPT_DIR = f"{ROOT_IN}/klein_ml_1L_ckpt_rescnn_multitime_stateplusphysics_stable"
CKPT_NAME = "final_model.pt"

EVAL_DIR = os.path.join(ROOT_OUT, "eval_rescnn_multitime_stateplusphysics")
os.makedirs(EVAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
G = 9.81
DT_MACRO = 200.0 * 30.0

Lx = 2.0e7
Ly = 8.0e6
NX = 256
NY = 128
DX = Lx / NX
DY = Ly / NY

# for geostrophic nudging
FMIN = 2.0e-5

# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
# MODE: "one", "list", "all"
MODE = "list"

ONE_CASE = "gauss_00"

CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
ROLLOUT_STEPS = 6

MAKE_FIELD_PLOTS = False
FIELD_PLOT_STEPS = [0, 2, 5]

# Which inference methods to test
RUN_PLAIN = True
RUN_DIFFUSION = True
RUN_GEONUDGE = True

# diffusion strengths
NU_LIST = [0.0, 5.0e4, 1.0e5, 2.0e5]

# geostrophic nudging strengths
ALPHA_LIST = [0.0, 0.05, 0.1, 0.2]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def domain_mean_ke(eta, u, v, H=1000.0):
    h = H + eta
    ke = 0.5 * h * (u*u + v*v)
    return float(np.mean(ke))

def list_ic_dirs(data_dir):
    return sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])

def list_ic_keys(data_dir):
    return [os.path.basename(d) for d in list_ic_dirs(data_dir)]

# ------------------------------------------------------------
# Discrete operators
# ------------------------------------------------------------
def laplacian_np(field, dx, dy):
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    d2x = (left - 2.0 * field + right) / (dx * dx)

    d2y = np.zeros_like(field)
    d2y[1:-1, :] = (field[2:, :] - 2.0 * field[1:-1, :] + field[:-2, :]) / (dy * dy)
    d2y[0, :]    = (field[1, :] - field[0, :]) / (dy * dy)
    d2y[-1, :]   = (field[-2, :] - field[-1, :]) / (dy * dy)

    return d2x + d2y

def ddx_np(field, dx):
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    return (right - left) / (2.0 * dx)

def ddy_np(field, dy):
    out = np.zeros_like(field)
    out[1:-1, :] = (field[2:, :] - field[:-2, :]) / (2.0 * dy)
    out[0, :]    = (field[1, :] - field[0, :]) / dy
    out[-1, :]   = (field[-1, :] - field[-2, :]) / dy
    return out

def safe_coriolis(f, fmin=2.0e-5):
    sign = np.sign(f)
    sign[sign == 0.0] = 1.0
    return sign * np.maximum(np.abs(f), fmin)

def geostrophic_velocity_from_eta(eta, f, dx, dy, g=9.81, fmin=2.0e-5):
    etax = ddx_np(eta, dx)
    etay = ddy_np(eta, dy)
    f_safe = safe_coriolis(f, fmin=fmin)
    ug = -(g / f_safe) * etay
    vg =  (g / f_safe) * etax
    return ug, vg

# ------------------------------------------------------------
# Exact residual CNN architecture from training
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, kernel_size=3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        x = self.stem(x)
        x = self.resnet(x)
        return x

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
ckpt_path = os.path.join(CKPT_DIR, CKPT_NAME)
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model = MultiStepContinuousResCNNModel().to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"[Eval] loaded checkpoint: {ckpt_path}")

# ------------------------------------------------------------
# Locate FD data
# ------------------------------------------------------------
data_dir = ckpt.get("data_dir", None)
if (data_dir is None) or (not os.path.exists(data_dir)):
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

print("[Eval] using FD data dir:", data_dir)

available_keys = list_ic_keys(data_dir)
print(f"[Eval] found {len(available_keys)} ICs")
print("[Eval] available ICs:", available_keys)

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------
if MODE == "one":
    selected_cases = [ONE_CASE]
elif MODE == "list":
    selected_cases = CASE_LIST[:]
elif MODE == "all":
    selected_cases = available_keys[:]
else:
    raise ValueError(f"Unknown MODE={MODE}")

selected_cases = [k for k in selected_cases if k in available_keys]
if len(selected_cases) == 0:
    raise RuntimeError("No valid selected cases found.")

print("[Eval] selected cases:", selected_cases)

# ------------------------------------------------------------
# Rollout variants
# ------------------------------------------------------------
def rollout_case(ic_key, mode_name="plain", nu=0.0, alpha=0.0):
    ic_dir = os.path.join(data_dir, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    n_avail = len(files) - START_INDEX - 1
    if n_avail <= 0:
        print(f"[Skip] {ic_key}: no rollout steps available.")
        return None

    rollout_steps = min(ROLLOUT_STEPS, n_avail)

    truth_seq = []
    truth_times = []
    truth_steps = []
    f_ref = None

    for fpath in files[START_INDEX:START_INDEX + rollout_steps + 1]:
        z = np.load(fpath)
        truth_seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
        truth_times.append(float(z["t"]))
        truth_steps.append(int(extract_step_from_path(fpath)))
        if f_ref is None:
            f_ref = z["f"].astype(np.float32)

    truth_seq = np.stack(truth_seq, axis=0)
    truth_times = np.array(truth_times, dtype=np.float32)
    truth_steps = np.array(truth_steps, dtype=np.int32)

    x = torch.tensor(truth_seq[0], dtype=torch.float32, device=device).unsqueeze(0)
    pred_seq = [truth_seq[0].copy()]

    with torch.no_grad():
        for k in range(rollout_steps):
            feat = model.encode(x)
            xdot = model.grid_tendency(feat)

            x_np = x[0].detach().cpu().numpy()
            xdot_np = xdot[0].detach().cpu().numpy()

            eta = x_np[0]
            u   = x_np[1]
            v   = x_np[2]

            eta_dot = xdot_np[0]
            u_dot   = xdot_np[1]
            v_dot   = xdot_np[2]

            eta_ml = eta + DT_MACRO * eta_dot
            u_ml   = u   + DT_MACRO * u_dot
            v_ml   = v   + DT_MACRO * v_dot

            if mode_name == "diffusion":
                u_ml = u_ml + DT_MACRO * nu * laplacian_np(u, DX, DY)
                v_ml = v_ml + DT_MACRO * nu * laplacian_np(v, DX, DY)

            elif mode_name == "geonudge":
                ug, vg = geostrophic_velocity_from_eta(
                    eta_ml, f_ref, DX, DY, g=G, fmin=FMIN
                )
                u_ml = (1.0 - alpha) * u_ml + alpha * ug
                v_ml = (1.0 - alpha) * v_ml + alpha * vg

            x_next_np = np.empty_like(x_np)
            x_next_np[0] = eta_ml
            x_next_np[1] = u_ml
            x_next_np[2] = v_ml

            x = torch.tensor(x_next_np, dtype=torch.float32, device=device).unsqueeze(0)
            pred_seq.append(x_next_np.copy())

    pred_seq = np.stack(pred_seq, axis=0)

    rmse_eta = []
    rmse_u = []
    rmse_v = []
    ke_fd = []
    ke_ml = []

    for k in range(1, rollout_steps + 1):
        eta_fd = truth_seq[k, 0]
        u_fd   = truth_seq[k, 1]
        v_fd   = truth_seq[k, 2]

        eta_ml = pred_seq[k, 0]
        u_ml   = pred_seq[k, 1]
        v_ml   = pred_seq[k, 2]

        rmse_eta.append(rmse(eta_ml, eta_fd))
        rmse_u.append(rmse(u_ml, u_fd))
        rmse_v.append(rmse(v_ml, v_fd))

        ke_fd.append(domain_mean_ke(eta_fd, u_fd, v_fd, H=H))
        ke_ml.append(domain_mean_ke(eta_ml, u_ml, v_ml, H=H))

    rmse_eta = np.array(rmse_eta)
    rmse_u   = np.array(rmse_u)
    rmse_v   = np.array(rmse_v)
    ke_fd    = np.array(ke_fd)
    ke_ml    = np.array(ke_ml)

    roll_steps = np.arange(1, rollout_steps + 1)
    roll_hours = roll_steps * DT_MACRO / 3600.0

    return {
        "ic_key": ic_key,
        "mode": mode_name,
        "nu": nu,
        "alpha": alpha,
        "rollout_steps": rollout_steps,
        "roll_hours": roll_hours,
        "truth_seq": truth_seq,
        "pred_seq": pred_seq,
        "rmse_eta": rmse_eta,
        "rmse_u": rmse_u,
        "rmse_v": rmse_v,
        "ke_fd": ke_fd,
        "ke_ml": ke_ml,
    }

# ------------------------------------------------------------
# Run evaluations
# ------------------------------------------------------------
summary_rows = []

for ic_key in selected_cases:
    print(f"\n[Eval] case: {ic_key}")
    case_results = []

    if RUN_PLAIN:
        result = rollout_case(ic_key, mode_name="plain")
        if result is not None:
            case_results.append(result)
            summary_rows.append({
                "ic_key": ic_key,
                "mode": "plain",
                "nu": "",
                "alpha": "",
                "final_eta_rmse": float(result["rmse_eta"][-1]),
                "final_u_rmse": float(result["rmse_u"][-1]),
                "final_v_rmse": float(result["rmse_v"][-1]),
                "final_fd_ke": float(result["ke_fd"][-1]),
                "final_ml_ke": float(result["ke_ml"][-1]),
                "final_ke_ratio": float(result["ke_ml"][-1] / max(result["ke_fd"][-1], 1e-12)),
            })
            print(
                f"  plain | eta={result['rmse_eta'][-1]:.4f} "
                f"u={result['rmse_u'][-1]:.4f} "
                f"v={result['rmse_v'][-1]:.4f} "
                f"KE ratio={result['ke_ml'][-1] / max(result['ke_fd'][-1], 1e-12):.4e}"
            )

    if RUN_DIFFUSION:
        for nu in NU_LIST:
            if nu == 0.0:
                continue
            result = rollout_case(ic_key, mode_name="diffusion", nu=nu)
            if result is not None:
                case_results.append(result)
                summary_rows.append({
                    "ic_key": ic_key,
                    "mode": "diffusion",
                    "nu": nu,
                    "alpha": "",
                    "final_eta_rmse": float(result["rmse_eta"][-1]),
                    "final_u_rmse": float(result["rmse_u"][-1]),
                    "final_v_rmse": float(result["rmse_v"][-1]),
                    "final_fd_ke": float(result["ke_fd"][-1]),
                    "final_ml_ke": float(result["ke_ml"][-1]),
                    "final_ke_ratio": float(result["ke_ml"][-1] / max(result["ke_fd"][-1], 1e-12)),
                })
                print(
                    f"  diff nu={nu:.1e} | eta={result['rmse_eta'][-1]:.4f} "
                    f"u={result['rmse_u'][-1]:.4f} "
                    f"v={result['rmse_v'][-1]:.4f} "
                    f"KE ratio={result['ke_ml'][-1] / max(result['ke_fd'][-1], 1e-12):.4e}"
                )

    if RUN_GEONUDGE:
        for alpha in ALPHA_LIST:
            if alpha == 0.0:
                continue
            result = rollout_case(ic_key, mode_name="geonudge", alpha=alpha)
            if result is not None:
                case_results.append(result)
                summary_rows.append({
                    "ic_key": ic_key,
                    "mode": "geonudge",
                    "nu": "",
                    "alpha": alpha,
                    "final_eta_rmse": float(result["rmse_eta"][-1]),
                    "final_u_rmse": float(result["rmse_u"][-1]),
                    "final_v_rmse": float(result["rmse_v"][-1]),
                    "final_fd_ke": float(result["ke_fd"][-1]),
                    "final_ml_ke": float(result["ke_ml"][-1]),
                    "final_ke_ratio": float(result["ke_ml"][-1] / max(result["ke_fd"][-1], 1e-12)),
                })
                print(
                    f"  geonudge a={alpha:.2f} | eta={result['rmse_eta'][-1]:.4f} "
                    f"u={result['rmse_u'][-1]:.4f} "
                    f"v={result['rmse_v'][-1]:.4f} "
                    f"KE ratio={result['ke_ml'][-1] / max(result['ke_fd'][-1], 1e-12):.4e}"
                )

    # save per-case plots
    if len(case_results) > 0:
        # RMSE comparison
        plt.figure(figsize=(9, 5))
        for result in case_results:
            hrs = result["roll_hours"]
            total_rmse = np.sqrt(result["rmse_eta"]**2 + result["rmse_u"]**2 + result["rmse_v"]**2)
            if result["mode"] == "plain":
                label = "plain"
            elif result["mode"] == "diffusion":
                label = f"diff nu={result['nu']:.1e}"
            else:
                label = f"geonudge a={result['alpha']:.2f}"
            plt.plot(hrs, total_rmse, label=label)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Combined RMSE")
        plt.title(f"RMSE comparison | {ic_key}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_rmse_compare.png"), dpi=160)
        plt.close()

        # KE comparison
        plt.figure(figsize=(9, 5))
        plt.plot(case_results[0]["roll_hours"], case_results[0]["ke_fd"], label="FD KE", linewidth=2)
        for result in case_results:
            hrs = result["roll_hours"]
            if result["mode"] == "plain":
                label = "ML plain"
            elif result["mode"] == "diffusion":
                label = f"ML diff nu={result['nu']:.1e}"
            else:
                label = f"ML geonudge a={result['alpha']:.2f}"
            plt.plot(hrs, result["ke_ml"], label=label)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Domain-mean kinetic energy")
        plt.title(f"KE comparison | {ic_key}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_ke_compare.png"), dpi=160)
        plt.close()

        # optional field plots for plain only
        if MAKE_FIELD_PLOTS:
            plain_result = None
            for result in case_results:
                if result["mode"] == "plain":
                    plain_result = result
                    break
            if plain_result is not None:
                truth_seq = plain_result["truth_seq"]
                pred_seq = plain_result["pred_seq"]

                for kk in FIELD_PLOT_STEPS:
                    if kk < 0 or kk >= plain_result["rollout_steps"]:
                        continue

                    t_idx = kk + 1
                    eta_fd = truth_seq[t_idx, 0]
                    u_fd   = truth_seq[t_idx, 1]
                    v_fd   = truth_seq[t_idx, 2]

                    eta_ml = pred_seq[t_idx, 0]
                    u_ml   = pred_seq[t_idx, 1]
                    v_ml   = pred_seq[t_idx, 2]

                    fig, axes = plt.subplots(3, 3, figsize=(14, 11))

                    im = axes[0, 0].imshow(eta_fd, origin="lower")
                    axes[0, 0].set_title(f"FD eta | step {t_idx}")
                    plt.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

                    im = axes[0, 1].imshow(eta_ml, origin="lower")
                    axes[0, 1].set_title("ML eta")
                    plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

                    im = axes[0, 2].imshow(eta_ml - eta_fd, origin="lower")
                    axes[0, 2].set_title("ML - FD eta")
                    plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

                    im = axes[1, 0].imshow(u_fd, origin="lower")
                    axes[1, 0].set_title("FD u")
                    plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

                    im = axes[1, 1].imshow(u_ml, origin="lower")
                    axes[1, 1].set_title("ML u")
                    plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

                    im = axes[1, 2].imshow(u_ml - u_fd, origin="lower")
                    axes[1, 2].set_title("ML - FD u")
                    plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

                    im = axes[2, 0].imshow(v_fd, origin="lower")
                    axes[2, 0].set_title("FD v")
                    plt.colorbar(im, ax=axes[2, 0], fraction=0.046, pad=0.04)

                    im = axes[2, 1].imshow(v_ml, origin="lower")
                    axes[2, 1].set_title("ML v")
                    plt.colorbar(im, ax=axes[2, 1], fraction=0.046, pad=0.04)

                    im = axes[2, 2].imshow(v_ml - v_fd, origin="lower")
                    axes[2, 2].set_title("ML - FD v")
                    plt.colorbar(im, ax=axes[2, 2], fraction=0.046, pad=0.04)

                    plt.tight_layout()
                    plt.savefig(os.path.join(EVAL_DIR, f"{ic_key}_fields_step_{t_idx:03d}.png"), dpi=150)
                    plt.close()

# ------------------------------------------------------------
# Save summary CSV
# ------------------------------------------------------------
if len(summary_rows) > 0:
    summary_csv = os.path.join(EVAL_DIR, "summary_eval.csv")
    keys = list(summary_rows[0].keys())
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in summary_rows:
            w.writerow(row)
    print("\n[Eval] wrote summary CSV:", summary_csv)
else:
    print("\n[Eval] no successful evaluations.")

Using device: cuda
GPU: Tesla T4
[Eval] loaded checkpoint: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_rescnn_multitime_stateplusphysics_stable/final_model.pt
[Eval] using FD data dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Eval] found 28 ICs
[Eval] available ICs: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
[Eval] selected cases: ['gauss_00', 'test_modon_00', 'test_rh_00', 'wave_00']

[Eval] case: gauss_00
  plain | eta=3.3232 u=8.5673 v=2.3740 KE ratio=4.4232e+02
  diff nu=5.0e+04 | eta=3.3214 u=8.5631 v=2.3714 KE ratio=4.4185e+02
  diff nu=1.0e+05 | eta=3.3202 u=8.5611 v=2.3700 KE ratio=4.4160e+02
  diff nu=2.0e+05 | eta=3.3187 u=8.5592 v=2.3684 KE ratio=4.4135e+02
  geonudge a=0.05 | eta=3.39

## New configurable script for residual cnn

In [ ]:
# ============================================================
# train_rescnn_1layer_configurable_ai4dvar2.py
# ------------------------------------------------------------
# Configurable residual-CNN trainer for 1-layer SWE emulator
# on Klein beta-plane, with:
#   - multistep rollout data loss
#   - collocation state loss
#   - collocation tendency loss
#   - continuity residual (flux form)
#   - momentum residual (nonlinear)
#   - weak geostrophic-balance loss
#   - weak damping / smoothness loss
#
# Outputs go to:
#   /content/drive/MyDrive/AI_4DVAR2/...
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------
class CFG:
    # -------- paths --------
    ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
    ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

    DATA_DIR_CANDIDATES = [
        f"{ROOT_IN}/klein_ckpt_1L",
        f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
        f"{ROOT_IN}/klein_ckpt_AL_centers",
        f"{ROOT_IN}/klein_ckpt_1L_colloc",
    ]
    COLLOC_DIR = f"{ROOT_IN}/klein_ml_1L/colloc_raw"

    # customize this tag per experiment
    EXP_NAME = "rescnn_b8_c96_lr1e4_t4_p4"

    # -------- domain / physics --------
    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 200.0 * 30.0
    FMIN = 2.0e-5

    # -------- architecture --------
    N_BLOCKS = 8
    FEAT_CH  = 96
    HIDDEN   = 192

    # -------- optimization --------
    EPOCHS = 12
    BATCH_SIZE = 1
    LR = 1e-4
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0

    # -------- rollout --------
    ROLL_STEPS = 4
    ROLL_GAMMA = 0.95

    # -------- data loading --------
    MAX_WINDOWS_PER_IC = 4
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # -------- collocation controls --------
    N_TIME_SLICES = 4
    PTS_PER_TIME  = 4

    # -------- loss weights --------
    LAMBDA_DATA       = 1.0
    LAMBDA_COLL_STATE = 0.05
    LAMBDA_COLL_TEND  = 0.1
    LAMBDA_CONT       = 0.2
    LAMBDA_MOM        = 0.5
    LAMBDA_GEO        = 0.01
    LAMBDA_SMOOTH     = 1e-4

    # -------- misc --------
    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 10
    RESUME_FROM = None

cfg = CFG()

cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
            "exp_name": cfg.EXP_NAME,
            "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom",
            "train_geo", "train_smooth"
        ])
        for row in loss_history:
            w.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        # zero-init safe start
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True,
        )
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        return state0_local + tau.unsqueeze(-1) * delta

# ------------------------------------------------------------
# Collocation sampling
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Derivative / balance helpers
# ------------------------------------------------------------
def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / cfg.Lx) - 1.0
    y_norm = (2.0 * y_m / cfg.Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = cfg.H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / cfg.Lx)
    eta_y = eta_yn * (2.0 / cfg.Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / cfg.Lx)
    u_y = u_yn * (2.0 / cfg.Ly)
    v_x = v_xn * (2.0 / cfg.Lx)
    v_y = v_yn * (2.0 / cfg.Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x = hu_xn * (2.0 / cfg.Lx)
    hv_y = hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    # state collocation
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state += ((u_hat - u_true) ** 2).mean()
    loss_coll_state += ((v_hat - v_true) ** 2).mean()

    # tendency collocation
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend += ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend += ((v_t - dvc_dt_fd) ** 2).mean()

    # continuity in flux form
    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    # nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    # weak geostrophic balance penalty
    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g =  (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    # weak smoothness / damping penalty
    loss_smooth = (u_x ** 2 + u_y ** 2 + v_x ** 2 + v_y ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom, loss_geo, loss_smooth

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)

    if not os.path.exists(cfg.COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {cfg.COLLOC_DIR}")

    colloc_bank = CollocBank(cfg.COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=cfg.ROLL_STEPS,
        max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel(
        feat_ch=cfg.FEAT_CH,
        hidden=cfg.HIDDEN,
        n_blocks=cfg.N_BLOCKS,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if cfg.RESUME_FROM is not None and os.path.exists(cfg.RESUME_FROM):
        print(f"[Resume] loading checkpoint: {cfg.RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(cfg.RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_BLOCKS              = {cfg.N_BLOCKS}")
    print(f"[Train] FEAT_CH               = {cfg.FEAT_CH}")
    print(f"[Train] LR                    = {cfg.LR}")
    print(f"[Train] N_TIME_SLICES         = {cfg.N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {cfg.PTS_PER_TIME}")
    print(f"[Train] colloc/interval       = {cfg.N_TIME_SLICES * cfg.PTS_PER_TIME}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0
        run_geo = 0.0
        run_smooth = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            loss_ctend = 0.0
            loss_cont = 0.0
            loss_mom = 0.0
            loss_geo = 0.0
            loss_smooth = 0.0
            n_phys = 0

            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + cfg.DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (cfg.ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=cfg.N_TIME_SLICES,
                        pts_per_time=cfg.PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_smooth = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=cfg.DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_smooth += l_smooth
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
                loss_geo = loss_geo / n_phys
                loss_smooth = loss_smooth / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero
                loss_geo = zero
                loss_smooth = zero

            loss = (
                cfg.LAMBDA_DATA       * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND  * loss_ctend
                + cfg.LAMBDA_CONT       * loss_cont
                + cfg.LAMBDA_MOM        * loss_mom
                + cfg.LAMBDA_GEO        * loss_geo
                + cfg.LAMBDA_SMOOTH     * loss_smooth
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  data   =", float(loss_data.detach().cpu()))
                print("  cstate =", float(loss_cstate.detach().cpu()))
                print("  ctend  =", float(loss_ctend.detach().cpu()))
                print("  cont   =", float(loss_cont.detach().cpu()))
                print("  mom    =", float(loss_mom.detach().cpu()))
                print("  geo    =", float(loss_geo.detach().cpu()))
                print("  smooth =", float(loss_smooth.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.detach().cpu())
            run_data += float(loss_data.detach().cpu())
            run_cstate += float(loss_cstate.detach().cpu())
            run_ctend += float(loss_ctend.detach().cpu())
            run_cont += float(loss_cont.detach().cpu())
            run_mom += float(loss_mom.detach().cpu())
            run_geo += float(loss_geo.detach().cpu())
            run_smooth += float(loss_smooth.detach().cpu())

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} "
                    f"data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} "
                    f"ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} "
                    f"mom={float(loss_mom.detach().cpu()):.6e} "
                    f"geo={float(loss_geo.detach().cpu()):.6e} "
                    f"smooth={float(loss_smooth.detach().cpu()):.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches
        ep_geo = run_geo / n_batches
        ep_smooth = run_smooth / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend, ep_cont, ep_mom, ep_geo, ep_smooth
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"geo={ep_geo:.6e} | smooth={ep_smooth:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cuda
GPU: Tesla T4
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | sampl